# Crawl review khách sạn Huế trên Agoda

## Bước 0: Cài đặt & Cấu hình

In [1]:
# Chạy 1 lần duy nhất nếu chưa cài
# !pip install selenium webdriver-manager beautifulsoup4 html5lib

import time
import re
from selenium import webdriver
from bs4 import BeautifulSoup

SO_KHACH_SAN = 10
SO_REVIEW_MOI_KS = 15
OUTPUT_FILE = "agoda_hue_reviews.txt"
CITY_URL = "https://www.agoda.com/vi-vn/city/hue-vn.html"

In [2]:
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)
print("Selenium sẵn sàng.")

Selenium sẵn sàng.


## Bước 1: Lấy danh sách top N khách sạn

In [3]:
driver.get(CITY_URL)
time.sleep(8)

driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html5lib")

hotel_links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/hotel/" in href:
        if href.startswith("/"):
            href = "https://www.agoda.com" + href
        if href not in hotel_links:
            hotel_links.append(href)

danh_sach_url = hotel_links[:SO_KHACH_SAN]
print("Số khách sạn lấy được:", len(danh_sach_url))
for u in danh_sach_url:
    print(u)

Số khách sạn lấy được: 10
https://www.agoda.com/vi-vn/angsana-lang-co/hotel/hue-vn.html
https://www.agoda.com/vi-vn/ana-mandara-hue-beach-resort/hotel/hue-vn.html
https://www.agoda.com/vi-vn/indochine-palace-hotel/hotel/hue-vn.html
https://www.agoda.com/vi-vn/imperial-hotel-hue/hotel/hue-vn.html
https://www.agoda.com/vi-vn/huong-giang-hotel-resort-spa/hotel/hue-vn.html
https://www.agoda.com/vi-vn/cherish-hue-hotel/hotel/hue-vn.html
https://www.agoda.com/vi-vn/moonlight-hotel-hue/hotel/hue-vn.html
https://www.agoda.com/vi-vn/century-riverside-hue-hotel/hotel/hue-vn.html
https://www.agoda.com/vi-vn/emm-hotel-hue_2/hotel/hue-vn.html
https://www.agoda.com/vi-vn/eldora-hotel/hotel/hue-vn.html


## Bước 2: Hàm cào 1 khách sạn

In [4]:
def cao_1_khach_san(hotel_url):
    driver.get(hotel_url)
    time.sleep(7)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, "html5lib")

    ten_el = soup.select_one('[data-selenium="hotel-header-name"]')
    if ten_el is not None:
        ten_co_so = ten_el.get_text(" ", strip=True)
    else:
        h1 = soup.find("h1")
        ten_co_so = h1.get_text(" ", strip=True) if h1 is not None else "Không tìm thấy"

    so_sao = ""
    star_box = soup.select_one('[data-element-name="mosaic-hotel-rating-container"]')
    if star_box is not None:
        star_text = star_box.get("aria-label", "")
        m = re.search(r"\d+(?:[.,]\d+)?", star_text)
        if m is not None:
            so_sao = m.group()

    diem_el = soup.select_one('[data-testid="review-plate-redesign-score"] h1')
    diem_tong = diem_el.get_text(strip=True) if diem_el is not None else "Không tìm thấy"

    review_blocks = soup.select("div.Review-comment")
    danh_sach_review = []
    for review in review_blocks[:SO_REVIEW_MOI_KS]:
        diem_el2 = review.select_one(".Review-comment-left span")
        diem = diem_el2.get_text(strip=True) if diem_el2 is not None else ""

        reviewer_el = review.select_one('[data-info-type="reviewer-name"]')
        quoc_tich = ""
        if reviewer_el is not None:
            spans = reviewer_el.find_all("span")
            quoc_tich = spans[-1].get_text(strip=True) if spans else ""

        di_theo_el = review.select_one('[data-info-type="group-name"]')
        di_theo = di_theo_el.get_text(" ", strip=True) if di_theo_el is not None else ""

        phong_el = review.select_one('[data-info-type="room-type"]')
        loai_phong = phong_el.get_text(" ", strip=True) if phong_el is not None else ""

        ngay_el = review.select_one(".Review-comment-bubble span")
        ngay = ngay_el.get_text(" ", strip=True) if ngay_el is not None else ""

        noi_dung_el = review.select_one(".Review-comment-bodyText")
        noi_dung = noi_dung_el.get_text(" ", strip=True) if noi_dung_el is not None else ""

        danh_sach_review.append({
            "noi_dung": noi_dung, "diem": diem, "ngay": ngay,
            "quoc_tich": quoc_tich, "loai_phong": loai_phong, "di_theo": di_theo,
        })

    return {"ten": ten_co_so, "so_sao": so_sao, "diem_tong": diem_tong, "reviews": danh_sach_review}

## Bước 3: Chạy toàn bộ và ghi ra file txt

In [5]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for idx, url in enumerate(danh_sach_url, start=1):
        print(f"Đang xử lý khách sạn {idx}/{len(danh_sach_url)}: {url}")
        try:
            data = cao_1_khach_san(url)

            f.write("===================================\n")
            f.write(f"KHÁCH SẠN {idx}: {data['ten']}\n")
            f.write("===================================\n")
            f.write(f"Số sao: {data['so_sao'] or 'không xác định'}\n")
            f.write(f"Điểm đánh giá tổng thể: {data['diem_tong']}\n\n")

            for r_idx, r in enumerate(data["reviews"], start=1):
                f.write(f"--- Review {r_idx} ---\n")
                f.write(f"Nội dung: {r['noi_dung']}\n")
                f.write(f"Điểm tổng thể: {r['diem']}\n")
                f.write(f"Thời điểm review: {r['ngay']}\n")
                f.write(f"Quốc tịch: {r['quoc_tich']}\n")
                f.write(f"Loại phòng: {r['loai_phong']}\n")
                f.write(f"Đi theo: {r['di_theo']}\n\n")

            print(f"  -> Lấy được {len(data['reviews'])} review")

        except Exception as e:
            print(f"  Lỗi: [{type(e).__name__}] {str(e)[:200]}")
            f.write(f"[LỖI khi crawl {url}]\n\n")
            continue

        time.sleep(2)

print("Xong. Kết quả lưu ở:", OUTPUT_FILE)

Đang xử lý khách sạn 1/10: https://www.agoda.com/vi-vn/angsana-lang-co/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 2/10: https://www.agoda.com/vi-vn/ana-mandara-hue-beach-resort/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 3/10: https://www.agoda.com/vi-vn/indochine-palace-hotel/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 4/10: https://www.agoda.com/vi-vn/imperial-hotel-hue/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 5/10: https://www.agoda.com/vi-vn/huong-giang-hotel-resort-spa/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 6/10: https://www.agoda.com/vi-vn/cherish-hue-hotel/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 7/10: https://www.agoda.com/vi-vn/moonlight-hotel-hue/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 8/10: https://www.agoda.com/vi-vn/century-riverside-hue-hotel/hotel/hue-vn.html
  -> Lấy được 5 review
Đang xử lý khách sạn 9/10: https://www.agoda.com/v

In [6]:
driver.quit()

## Bước 4: Kiểm tra nhanh kết quả

In [7]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    print(f.read()[:2000])

KHÁCH SẠN 1: Angsana Lang Co
Số sao: 5
Điểm đánh giá tổng thể: 8,8

--- Review 1 ---
Nội dung: Khách sạn rất đẹp, trang bị phòng ốc đầy đủ, tiện nghi, nhiều cây, không khí trong lành, yên tĩnh. Bể bơi sạch sẽ ngay trong khuôn viên khách sạn, bãi biển riêng biệt, sạch sẽ, có nhiều dịch vụ. Ăn sáng nhiều món, thực phẩm được chế biến đa dạng, bảo đảm vệ sinh an toàn.
Đội ngũ nhân viên thân thiện lễ phép. Tuy nhiên, do khách sạn ở nơi tương đối biệt lập nên du khách ít có điều kiện giao lưu và thăm thú bên ngoài. Địa điểm này cách Đà Nẵng 60km và cách Huế 56km nên di chuyển chỉ có xe gia đình hoặc taxi, nói chung không được tiện lắm.
Trong khu sauna có phòng xông ướt và xông khô miễn phí, được thiết kế điều khiển tự động, khách sạn nên có biển hướng dẫn cách sử dụng hoặc có nhân viên thường trực để hỗ trợ du khách.
Điểm tổng thể: 10,0
Thời điểm review: Đã nhận xét vào 13 tháng 6 2026
Quốc tịch: Việt Nam
Loại phòng: Phòng Grand hướng vườn 2 giường có ban công
Đi theo: Nhóm

--- Review 2 ---